In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
def f(x):
    return x ** 2

In [ ]:
f(3.0)

In [ ]:
x_values = np.arange(-5,5.25,0.25)
y_values = f(x_values)

In [ ]:
y_values

In [ ]:
plt.plot(x_values, y_values)

In [ ]:
h = 0.000000001
a = 2.0
b = -3.0
c = 10.0
d1 = a*b + c
d2 = (a)*(b) + (c+h)
slope = (d2 - d1)/h
print(f'd1: {d1}; d2: {d2}')
print(slope)

In [ ]:
class Value:
  
  def __init__(self, data, _children=(), _op='', label=''):
    self.data = data
    self.grad = 0.0
    self._backward = lambda: None
    self._prev = set(_children)
    self._op = _op
    self.label = label

  def __repr__(self):
    return f"Value(data={self.data})"
  
  def __add__(self, other):
    other = other if isinstance(other, Value) else Value(other)
    out = Value(self.data + other.data, (self, other), '+')
    
    def _backward():
      self.grad += 1.0 * out.grad
      other.grad += 1.0 * out.grad
    out._backward = _backward
    
    return out

  def __mul__(self, other):
    other = other if isinstance(other, Value) else Value(other)
    out = Value(self.data * other.data, (self, other), '*')
    
    def _backward():
      self.grad += other.data * out.grad
      other.grad += self.data * out.grad
    out._backward = _backward
      
    return out
  
  def __pow__(self, other):
    assert isinstance(other, (int, float)), "only supporting int/float powers for now"
    out = Value(self.data**other, (self,), f'**{other}')

    def _backward():
        self.grad += other * (self.data ** (other - 1)) * out.grad
    out._backward = _backward

    return out
  
  def __rmul__(self, other): # other * self
    return self * other

  def __truediv__(self, other): # self / other
    return self * other**-1

  def __neg__(self): # -self
    return self * -1

  def __sub__(self, other): # self - other
    return self + (-other)

  def __radd__(self, other): # other + self
    return self + other

  def tanh(self):
    x = self.data
    t = (math.exp(2*x) - 1)/(math.exp(2*x) + 1)
    out = Value(t, (self, ), 'tanh')
    
    def _backward():
      self.grad += (1 - t**2) * out.grad
    out._backward = _backward
    
    return out
  
  def exp(self):
    x = self.data
    out = Value(math.exp(x), (self, ), 'exp')
    
    def _backward():
      self.grad += out.data * out.grad # NOTE: in the video I incorrectly used = instead of +=. Fixed here.
    out._backward = _backward
    
    return out
  
  
  def backward(self):
    
    topo = []
    visited = set()
    def build_topo(v):
      if v not in visited:
        visited.add(v)
        for child in v._prev:
          build_topo(child)
        topo.append(v)
    build_topo(self)
    
    self.grad = 1.0
    for node in reversed(topo):
      node._backward()

In [ ]:
a = Value(2.0, label="a")
b = Value(-3.0, label="b")
c = Value(10.0, label="c")
e = a*b; e.label = "e"
d = e + c; d.label = "d"
f = Value(-2.0, label="f")
L = d * f; L.label="L"


In [ ]:
from graphviz import Digraph

def trace(root):
  # builds a set of all nodes and edges in a graph
  nodes, edges = set(), set()
  def build(v):
    if v not in nodes:
      nodes.add(v)
      for child in v._prev:
        edges.add((child, v))
        build(child)
  build(root)
  return nodes, edges

def draw_dot(root):
  dot = Digraph(format='svg', graph_attr={'rankdir': 'LR'}) # LR = left to right
  
  nodes, edges = trace(root)
  for n in nodes:
    uid = str(id(n))
    # for any value in the graph, create a rectangular ('record') node for it
    dot.node(name = uid, label = "{ %s | data %.4f | grad %.4f }" % (n.label, n.data, n.grad), shape='record')
    if n._op:
      # if this value is a result of some operation, create an op node for it
      dot.node(name = uid + n._op, label = n._op)
      # and connect this node to it
      dot.edge(uid + n._op, uid)

  for n1, n2 in edges:
    # connect n1 to the op node of n2
    dot.edge(str(id(n1)), str(id(n2)) + n2._op)

  return dot

In [ ]:
draw_dot(L)

In [ ]:
# iterate through all nodes in reverse and locally apply the chain rule, to calculate the gradient of L with respect to each node
# this is the fundementals of backpropagation
L.grad = 1.0 
# dL/dd is f, dL/df is d (calculus)
d.grad = -2.0
f.grad = 4.0
# dd/de = dd/dc = 1 (calculus)
# use chain rule to get dL/de and dL/dc
# dL/de = dL/dd * dd/de 
e.grad = -2.0
# dL/dc = dL/dd * dd/dc 
c.grad = -2.0
# find dL/da and dL/db
# dL/da = dL/de * de/da
a.grad = 6.0
# dL/db = dL/de * de/db
b.grad = -4.0


In [ ]:
def derviative():
    h = 0.0001

    a = Value(2.0, label="a")
    b = Value(-3.0, label="b")
    c = Value(10.0, label="c")
    e = a*b; e.label = "e"
    d = e + c; d.label = "d"
    f = Value(-2.0, label="f")
    L = d * f; L.label="L"
    L1 = L.data

    a = Value(2.0 + h, label="a") # dL/da
    b = Value(-3.0, label="b")
    c = Value(10.0, label="c")
    e = a*b; e.label = "e"
    d = e + c; d.label = "d"
    f = Value(-2.0, label="f")
    L = d * f; L.label="L"
    L2 = L.data

    print((L2-L1)/h)


In [ ]:
derviative()

In [ ]:
plt.plot(np.arange(-5,5,0.2), np.tanh(np.arange(-5,5,0.2))); plt.grid();

backpropagation ex2: neural network

In [ ]:
# inputs x1,x2
x1 = Value(2.0, label='x1')
x2 = Value(0.0, label='x2')
# weights w1,w2
w1 = Value(-3.0, label='w1')
w2 = Value(1.0, label='w2')
# bias of the neuron
b = Value(6.8813735870195432, label='b')
# x1*w1 + x2*w2 + b
x1w1 = x1*w1; x1w1.label = 'x1*w1'
x2w2 = x2*w2; x2w2.label = 'x2*w2'
x1w1x2w2 = x1w1 + x2w2; x1w1x2w2.label = 'x1*w1 + x2*w2'
n = x1w1x2w2 + b; n.label = 'n'
o = n.tanh(); o.label = 'o' # tanh is a squashing function so that the values of the nodes don't explode


In [ ]:
draw_dot(o)

In [ ]:
# lets not do backpropagation manually lol
o.grad = 1.0

In [ ]:
o._backward()
n._backward()
b._backward()
x1w1x2w2._backward()
x1w1._backward()
x2w2._backward()


In [ ]:
# better, but we still call the function manually 

# def backward(self):
    
#     topo = []
#     visited = set()
#     def build_topo(v):
#       if v not in visited:
#         visited.add(v)
#         for child in v._prev:
#           build_topo(child)
#         topo.append(v)
#     build_topo(self)
    
#     self.grad = 1.0
#     for node in reversed(topo):
#       node._backward()

# hidden in Value class - Use topological sort to make each node only get added to the list after its child nodes were already added
# this causes the bias node to be added first (leaf node) and o node to be added last (most children).
# since we are doing backpropgation, reverse the list and apply ._backward() to each node 

In [ ]:
o.backward()

In [ ]:
draw_dot(o)

In [ ]:
# work with what you have already, implement more complex operations by breaking them up into already implemented simpler ones (peep formula for tanh)

# Working with PyTorch


In [ ]:
import torch
import random

In [ ]:
x1 = torch.Tensor([2.0]).double()                ; x1.requires_grad = True # .double to convert to float64 (like in python)
x2 = torch.Tensor([0.0]).double()                ; x2.requires_grad = True
w1 = torch.Tensor([-3.0]).double()               ; w1.requires_grad = True
w2 = torch.Tensor([1.0]).double()                ; w2.requires_grad = True
b = torch.Tensor([6.8813735870195432]).double()  ; b.requires_grad = True
n = x1*w1 + x2*w2 + b
o = torch.tanh(n)

print(o.data.item())
o.backward()

print('---')
print('x2', x2.grad.item())
print('w2', w2.grad.item())
print('x1', x1.grad.item())
print('w1', w1.grad.item())

# Let's make a Neural Network!

In [ ]:
class Neuron:
    def __init__(self, nin): # nin: number of inputs
        self.w = [Value(random.uniform(-1,1)) for _ in range(nin)] # weight
        self.b = Value(random.uniform(-1,1)) # bias
    
    def __call__(self, x):
        act = sum((wi*xi for wi, xi in zip(self.w,x)), self.b)
        out = act.tanh()
        return out

    def parameters(self):
        return self.w + [self.b]

class Layer: # layer of neurons
  
  def __init__(self, nin, nout): # nout: how many neurons in layer
    self.neurons = [Neuron(nin) for _ in range(nout)]
  
  def __call__(self, x):
    outs = [n(x) for n in self.neurons]
    return outs[0] if len(outs) == 1 else outs
  
  def parameters(self):
    return [p for neuron in self.neurons for p in neuron.parameters()]

class MLP: # Multi Layer Perceptron: NN
  
  def __init__(self, nin, nouts): # nouts: number of layers of the MLP
    sz = [nin] + nouts
    self.layers = [Layer(sz[i], sz[i+1]) for i in range(len(nouts))]
  
  def __call__(self, x):
    for layer in self.layers:
      x = layer(x)
    return x
  
  def parameters(self):
    return [p for layer in self.layers for p in layer.parameters()]

In [ ]:
x = [2.0, 3.0, -1.0]
n = MLP(3, [4, 4, 1])
# this is a MLP with 3 inputs, 2 layers of neurons and 1 output layer
n(x) 

In [ ]:
draw_dot(n(x))

In [ ]:
# small dataset 
xs = [
  [2.0, 3.0, -1.0],
  [3.0, -1.0, 0.5],
  [0.5, 1.0, 1.0],
  [1.0, 1.0, -1.0],
]
ys = [1.0, -1.0, -1.0, 1.0] # desired targets
ypred = [n(x) for x in xs]
ypred # not very good yet

In [ ]:
# tune the weights by calculating a number that measures the total performance of the NN: the Loss Function
loss = [(yout - ygt)**2 for ygt, yout in zip(ys, ypred)] # MSE for each output
loss

In [ ]:
loss = sum((yout - ygt)**2 for ygt, yout in zip(ys, ypred)) # we want to minimize loss - all outputs as close as possible to their targets  
loss.backward() 
n.layers[0].neurons[0].w[0].grad 

In [ ]:
draw_dot(loss) # 4 passes of MLP: huge graph

In [ ]:
# Gradient descent: think of the gradient as a vector
# We modify the value of the node in the opposite direction of the gradient by a small step (proportional to the gradient) - reduces loss
for p in n.parameters():
    p.data += -0.01 * p.grad

Final code

In [ ]:
xs = [
  [2.0, 3.0, -1.0],
  [3.0, -1.0, 0.5],
  [0.5, 1.0, 1.0],
  [1.0, 1.0, -1.0],
]
ys = [1.0, -1.0, -1.0, 1.0] # desired targets

In [ ]:
for k in range(50):
  
  # forward pass
  ypred = [n(x) for x in xs]
  loss = sum((yout - ygt)**2 for ygt, yout in zip(ys, ypred))
  
  # backward pass
  for p in n.parameters():
    p.grad = 0.0
  loss.backward()
  
  # update
  for p in n.parameters():
    p.data += -0.1 * p.grad
  
  print(k, loss.data)